# **Pruning with a modified Cambrige-Aachen algorithm**

In order to perform the PDF pruning only based on the PDFs shapes we use the $k_t$-like algorithm with $p=0$, which implies that the algoritm is based only on the geometric information [doi: 10.1007/s100520050232.]. To account the mutual distance between any two PDF replicas (vectors) $\mathbf{v}_i$ and $\mathbf{v}_j$, we define a Dissimilarity  metric as the absolute difference normalized by the average of the absolute values between each vector (This metric is also know as the Symmetric mean absolute percentage error (SMAPE)), i.e.,

$$
    D(\mathbf{v}_i, \mathbf{v}_j) =
\begin{cases} 
\sum_{k=1}^{N} 2 \dfrac{|v_{i,k} - v_{j,k}|}{|v_{i,k}| + |v_{j,k}|} & \text{if } |v_{i,k}| + |v_{j,k}| > \epsilon \\
0 & \text{if } |v_{i,k}| + |v_{i,j}| \leq \epsilon
\end{cases}
$$

where $\epsilon$ is a tolerance value (in our case $\epsilon=0.0001$). This tolerance value defines when a difference between PDFs is acceptable or is taken as a numerical noise, it ensures numerical stability and focus on physically significant regions

Then, the Cambrige-Aachen algorithm to find similar PDFs becomes,

$$
    d_{ij}=\dfrac{D(\mathbf{v}_i, \mathbf{v}_j)^2}{R^2}, ~~ d_{B}=1
$$

where $R$ is a hyperparemter that defines the set size of the PDFs that are consider similar. Also, the recombination scheme is taken as
$$
    \mathbf{v}_{new}=\dfrac{ \mathbf{v}_i+ \mathbf{v}_j }{2}
$$



# Packages

In [1]:
import pruning_tools as pt
import pandas as pd
from pathlib import Path

#  1.0 Loading Data

In [2]:
# Loading vectors
data_folder = Path("260211_data")
vec_filename = "260211.PN.pq206p357_nx22.tsv"

df_vectors = pd.read_csv(data_folder / vec_filename, sep='\t', header=None)

df_vectors

,0,1,2,3,4,5,6,7,8,9,...,166,167,168,169,170,171,172,173,174,175
0,5.260367e+02,386.415486,269.408098,184.448681,125.269443,84.635160,56.886933,37.996947,25.179845,16.524622,...,11.545765,7.479656,4.700648,2.822296,1.588127,0.817065,0.371822,0.142520,0.042204,0.007997
1,2.443227e-311,358.566569,240.694529,168.295338,118.182774,81.838675,55.882162,37.669578,25.077153,16.482733,...,11.537903,7.441649,4.646927,2.763080,1.528989,0.761733,0.325279,0.110382,0.026368,0.003601
2,0.000000e+00,195.503507,272.381928,188.841528,126.410092,84.442234,56.500968,37.797378,25.197590,16.678918,...,11.150080,7.203013,4.533929,2.742641,1.564442,0.817450,0.373065,0.136650,0.034437,0.004600
3,0.000000e+00,157.578495,187.739257,143.582829,104.943319,75.038411,52.860471,36.752428,25.213445,17.046484,...,11.745199,7.642875,4.805036,2.865864,1.583709,0.786125,0.336576,0.117783,0.031213,0.005451
4,0.000000e+00,13.204578,198.220385,170.072635,119.142958,80.712394,54.359956,36.558917,24.525352,16.367435,...,11.408389,7.399302,4.671581,2.826493,1.602314,0.822154,0.361551,0.125163,0.030180,0.004300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,0.000000e+00,368.712027,300.283523,202.159778,138.478551,93.939128,62.760880,41.249158,26.656865,16.932183,...,11.398366,7.492935,4.759615,2.867438,1.600204,0.804465,0.353723,0.131613,0.038871,0.007607
353,8.052539e-312,259.147027,194.201087,142.884285,104.494585,75.255453,53.459760,37.494618,25.949265,17.688032,...,11.840744,7.738466,4.900939,2.954140,1.654262,0.832198,0.358424,0.123052,0.030171,0.004380
354,5.027101e-311,665.733283,372.786560,226.293493,143.637164,92.604661,60.096315,39.048224,25.294868,16.275025,...,11.792011,7.734593,4.927044,2.996796,1.699485,0.867458,0.377816,0.129231,0.030295,0.003887
355,1.867745e-317,384.640557,323.978002,206.550629,132.523629,85.852817,56.053541,36.764975,24.137967,15.806654,...,11.833362,7.629937,4.740700,2.785137,1.511685,0.737306,0.313186,0.111874,0.032119,0.006721


In [3]:
met_filename = "260211.PN.pq206p357_nx22_metadata.tsv"
metadata = pd.read_csv(data_folder / met_filename, sep="\t", usecols=[0,1])
metadata

,VID,cluster_label
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN
...,...,...
352,353,NaN
353,354,NaN
354,355,NaN
355,356,NaN


# 2.0 **Clustering Analisis**

## 2.1 Neighborhood Degree and Pairwise Distance Analysis

The $R$ parameter is the only hyperparameter on this algoritm and it defines the set size of the PDFs that are consider similar. Therefore for pruning purposes we should take $R$ enough to include in a same family only the replicas that are really close.

To determine an appropriate distance threshold empirically, we computed all pairwise PDF dissimilarities for the vector set with $n$ replicas. The resulting figure shows the Cumulative Distribution Function (CDF), which represents the fraction of replica pairs separated by a distance less than or equal to a given value $R$.

In [4]:
vec = df_vectors.values
plt_cdf = pt.plot_pairwise_cdf(data_input=vec, R_cutoff=2, metric=pt.pdf_dissimilarity)
plt_cdf.show()  

────────────────────────────────────────────────────
  Pairwise Distance Statistics  (Custom Metric)
────────────────────────────────────────────────────
  Vectors (N)        : 357
  Pairs  N(N-1)/2    : 63,546

  Min distance       : 0.1971
  Mean distance      : 28.1804
  Median distance    : 28.2526
  Std deviation      : 7.4593
  Max distance       : 55.8513

  Cutoff  R = 2
    R is 3.52 std devs below the median

  Pairs with d ≤ R   : 8 / 63,546
  → Merged (pruned)  : 0.01%   [CDF(2) = 0.0001]
  → Unmerged         : 99.99%
────────────────────────────────────────────────────


In this case R=2 will merge only vectors that are 1% 

In [5]:
pt.plot_neighbor_degree(data_input=df_vectors, R_cutoff=2.0, metric=pt.pdf_dissimilarity )

────────────────────────────────────────────────────
  Neighbor degree  (R = 2.0, Custom Metric)
────────────────────────────────────────────────────
  N vectors                : 357
  Isolated  (k=0)          : 341  (95.5%)
    → guaranteed solo clusters
  With neighbors (k≥1)     : 16  (4.5%)
  Mean degree              : 0.04
  Median degree            : 0.0
  Max degree (biggest hub) : 1
  Total close pairs        : 8
────────────────────────────────────────────────────


In [6]:
pt.neighbor_degree_analysis(data_input=df_vectors, R_cutoff=2.0, metric=pt.pdf_dissimilarity)

────────────────────────────────────────────────────
  Neighbor degree analysis  (R = 2.0)
────────────────────────────────────────────────────
  Vectors (N)              : 357

  Isolated  (0 neighbors)  : 341
  → These will ALWAYS form solo clusters, regardless of R.

  With >= 1 neighbor       : 16
  → These are candidates for merging.

  Min neighbors            : 0
  Max neighbors            : 1  ← hub size
  Mean neighbors           : 0.04
  Median neighbors         : 0.0

  Vectors above mean       : 16  ← potential cluster cores

  Pairs check (sum/2)      : 8
  → Should match n_pairs_below_cutoff from plot_pairwise_cdf.

  Predicted clusters (rough estimate)
  → Lower bound (aggressive): 341 isolated + 16 hubs = ~357
  → Upper bound              = N = 357  (no merging at all)
────────────────────────────────────────────────────


{'counts': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        1, 1, 0, 0, 0, 0, 0,

## 2.2 Clustering search

In [7]:
#clustering serach
vec = df_vectors.values
clstr = pt.cambridge_cluster(vectors=vec, R=2.0, metric=pt.pdf_dissimilarity)


In [8]:
clstr_results = pd.DataFrame(clstr).sort_values(by='num_vectors', ascending=False)
clstr_results

,cluster_label,vectors_in_cluster,cluster_vector,num_vectors,closest_vec_to_centroid_id,dist_of_closest_to_centroid
259,259,"[263, 285]","[660.4716137422929, 438.7579236956458, 288.106...",2,285,421.803106
124,124,"[124, 127]","[632.5095713642245, 426.48453303890426, 282.55...",2,124,195.348156
205,205,"[208, 210]","[1067.0913387982723, 505.5097357824799, 288.02...",2,210,78.386442
150,150,"[151, 165]","[1114.1342842978456, 521.2518549218335, 294.42...",2,151,477.176397
270,270,"[275, 287]","[1175.96112685722, 546.7579400059942, 304.9197...",2,275,213.298912
...,...,...,...,...,...,...
112,112,[112],"[0.0, 550.0867055110899, 385.66033591097846, 2...",1,112,0.000000
111,111,[111],"[0.0, 266.8784328636043, 271.8686598127716, 20...",1,111,0.000000
110,110,[110],"[0.0, 265.95319087949025, 382.54105972444, 239...",1,110,0.000000
109,109,[109],"[0.0, 306.75177952036296, 353.80645577869774, ...",1,109,0.000000


## 2.3 New metadata, best and discard vectors

In [18]:
# New Metadata
new_metadata = pt.assign_cluster_data_to_metadata(final_jets=clstr, cluster_best_field="closest_vec_to_centroid_id")
new_metadata

,closest_vec_to_centroid_id,cluster_label,cluster_size,cluster_best_idx,is_best,dist_of_closest_to_centroid
vector_id,,,,,,
0,0,0,1,0,True,0.0
1,1,1,1,1,True,0.0
2,2,2,1,2,True,0.0
3,3,3,1,3,True,0.0
4,4,4,1,4,True,0.0
...,...,...,...,...,...,...
352,352,344,1,352,True,0.0
353,353,345,1,353,True,0.0
354,354,346,1,354,True,0.0


In [17]:
best_vectors, discarded_vectors = pt.get_pruned_and_discarded_vectors(vectors_df=df_vectors, labeled_metadata=new_metadata, metric_column="closest_vec_to_centroid_id")

--- Pruning Summary (best by: closest_vec_to_centroid_id) ---
Total Original : 357
Kept (Best)    : 349
Discarded      : 8
Reduction      : 2.2%


In [13]:
best_vectors.to_csv(data_folder / (vec_filename + "_best_replicas.tsv"), sep="\t", index=False)
best_vectors

,0,1,2,3,4,5,6,7,8,9,...,166,167,168,169,170,171,172,173,174,175
0,5.260367e+02,386.415486,269.408098,184.448681,125.269443,84.635160,56.886933,37.996947,25.179845,16.524622,...,11.545765,7.479656,4.700648,2.822296,1.588127,0.817065,0.371822,0.142520,0.042204,0.007997
1,2.443227e-311,358.566569,240.694529,168.295338,118.182774,81.838675,55.882162,37.669578,25.077153,16.482733,...,11.537903,7.441649,4.646927,2.763080,1.528989,0.761733,0.325279,0.110382,0.026368,0.003601
2,0.000000e+00,195.503507,272.381928,188.841528,126.410092,84.442234,56.500968,37.797378,25.197590,16.678918,...,11.150080,7.203013,4.533929,2.742641,1.564442,0.817450,0.373065,0.136650,0.034437,0.004600
3,0.000000e+00,157.578495,187.739257,143.582829,104.943319,75.038411,52.860471,36.752428,25.213445,17.046484,...,11.745199,7.642875,4.805036,2.865864,1.583709,0.786125,0.336576,0.117783,0.031213,0.005451
4,0.000000e+00,13.204578,198.220385,170.072635,119.142958,80.712394,54.359956,36.558917,24.525352,16.367435,...,11.408389,7.399302,4.671581,2.826493,1.602314,0.822154,0.361551,0.125163,0.030180,0.004300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,0.000000e+00,368.712027,300.283523,202.159778,138.478551,93.939128,62.760880,41.249158,26.656865,16.932183,...,11.398366,7.492935,4.759615,2.867438,1.600204,0.804465,0.353723,0.131613,0.038871,0.007607
353,8.052539e-312,259.147027,194.201087,142.884285,104.494585,75.255453,53.459760,37.494618,25.949265,17.688032,...,11.840744,7.738466,4.900939,2.954140,1.654262,0.832198,0.358424,0.123052,0.030171,0.004380
354,5.027101e-311,665.733283,372.786560,226.293493,143.637164,92.604661,60.096315,39.048224,25.294868,16.275025,...,11.792011,7.734593,4.927044,2.996796,1.699485,0.867458,0.377816,0.129231,0.030295,0.003887
355,1.867745e-317,384.640557,323.978002,206.550629,132.523629,85.852817,56.053541,36.764975,24.137967,15.806654,...,11.833362,7.629937,4.740700,2.785137,1.511685,0.737306,0.313186,0.111874,0.032119,0.006721


In [14]:
discarded_vectors.to_csv( data_folder / (vec_filename + "_discarded_replicas.tsv"), sep="\t", index=False)
discarded_vectors

,0,1,2,3,4,5,6,7,8,9,...,166,167,168,169,170,171,172,173,174,175
127,633.453761,426.634405,282.537005,187.245853,124.570649,83.129810,55.533969,37.047184,24.615488,16.245816,...,11.572292,7.448459,4.678555,2.832104,1.622357,0.855097,0.396509,0.150244,0.040951,0.006344
165,1105.421461,511.077909,288.929418,180.543817,118.883591,80.286601,54.667466,37.109759,24.930738,16.502890,...,11.649953,7.544773,4.744773,2.855416,1.613690,0.834928,0.381527,0.145570,0.041860,0.007164
178,591.727618,450.286909,295.961011,190.115873,123.485030,81.546083,54.507060,36.606456,24.518626,16.276939,...,12.119039,7.759703,4.809418,2.856945,1.605263,0.835342,0.387759,0.149727,0.042093,0.006596
208,1080.615117,516.831254,293.837620,182.350054,118.767058,79.426723,53.783582,36.514731,24.669909,16.487039,...,11.701807,7.580311,4.761960,2.856717,1.604692,0.822433,0.371242,0.140120,0.040391,0.007179
263,659.993669,438.536212,288.012567,189.797536,125.754998,83.665820,55.763217,37.135070,24.642010,16.249066,...,11.819979,7.696884,4.860887,2.928912,1.647958,0.840444,0.371947,0.133367,0.034445,0.005213
264,689.863891,497.014726,310.665484,192.915040,123.037510,80.721061,54.000305,36.424735,24.517408,16.332763,...,11.824660,7.670837,4.801642,2.860207,1.596017,0.818413,0.375862,0.148563,0.047008,0.010138
274,1179.741484,549.116874,306.386246,187.489345,120.788583,80.083501,53.858348,36.371787,24.477690,16.317451,...,11.676480,7.528302,4.714719,2.829549,1.599640,0.832338,0.385357,0.150072,0.043998,0.007440
287,1179.660476,547.862161,305.301411,186.731924,120.322451,79.840459,53.772222,36.385308,24.544347,16.402066,...,11.849446,7.699949,4.848668,2.911439,1.632497,0.830615,0.368033,0.133461,0.036033,0.006514


## 2.4 Cluster serparation

$$\text{Separation Ratio}_i = \frac{\text{Avg\_Inter\_Dist}_i}{\text{Avg\_Intra\_Dist}_i + \epsilon}$$

Where:
 * $\text{Avg\_Intra\_Dist}_i$ (Tightness): The average distance from the members of cluster $i$ to their representative centroid.
 * $\text{Avg\_Inter\_Dist}_i$ 
 (Separation): The average distance from the centroid of cluster $i$ to the centroids of all other clusters.
 * $\epsilon$ ($1e-9$): A tiny constant added to the denominator to prevent "Division by Zero" errors if a cluster contains only one vector.


Ideally, intra-cluster distances should be small while inter-cluster distances are large, indicating well-separated, cohesive clusters.

In [15]:
# Distances beteween the centroid and the internal elements of the cluster (Intra_Dist)
# Distances between each centroid of each cluster (Inter_Dist)
cluster_sep=pt.analyze_cluster_separation(clstr, df_vectors)
cluster_sep

,Cluster,Size,Avg_Intra_Dist,Avg_Inter_Dist,Separation_Ratio
0,0,1,0.0,9434.207400,9.434207e+12
1,1,1,0.0,10869.853257,1.086985e+13
2,2,1,0.0,11355.526245,1.135553e+13
3,3,1,0.0,9773.597191,9.773597e+12
4,4,1,0.0,9782.492986,9.782493e+12
...,...,...,...,...,...
344,344,1,0.0,13163.216622,1.316322e+13
345,345,1,0.0,13136.041886,1.313604e+13
346,346,1,0.0,9524.521896,9.524522e+12
347,347,1,0.0,9401.069291,9.401069e+12


In [16]:
pltcluster_sep = pt.plot_cluster_separation(cluster_sep)
pltcluster_sep.show()